In [ ]:
import cv2
import numpy as np
import os
import time
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from deep_sort_realtime.deepsort_tracker import DeepSort
from ultralytics import YOLO

# Configuration
INPUT_DIR = "content"
OUTPUT_DIR = "content/processed/stress_tests"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CONF_THRESHOLD = 0.50
FRAME_LIMIT = 150

# Synthetic Degradation Functions
def apply_darkness(frame, severity=0.5):
    factor = 1.0 - severity
    darkened = (frame.astype(np.float32) * factor).clip(0, 255).astype(np.uint8)
    return darkened

def apply_fog(frame, severity=0.5):
    atmospheric_light = np.ones_like(frame) * 200
    fogged = cv2.addWeighted(frame, 1.0 - severity, 
                             atmospheric_light.astype(np.uint8), severity, 0)
    return fogged

def apply_noise(frame, severity=0.5):
    noise = np.random.normal(0, severity * 50, frame.shape).astype(np.float32)
    noisy = (frame.astype(np.float32) + noise).clip(0, 255).astype(np.uint8)
    return noisy

def apply_motion_blur(frame, severity=0.5):
    kernel_size = int(severity * 15) + 1
    if kernel_size % 2 == 0:
        kernel_size += 1
    kernel = np.zeros((kernel_size, kernel_size))
    kernel[int((kernel_size-1)/2), :] = np.ones(kernel_size)
    kernel = kernel / kernel_size
    blurred = cv2.filter2D(frame, -1, kernel)
    return blurred

def apply_combined_degradation(frame, darkness=0, fog=0, noise=0, blur=0):
    degraded = frame.copy()
    if darkness > 0: degraded = apply_darkness(degraded, darkness)
    if fog > 0: degraded = apply_fog(degraded, fog)
    if noise > 0: degraded = apply_noise(degraded, noise)
    if blur > 0: degraded = apply_motion_blur(degraded, blur)
    return degraded

# Enhancement Functions
def get_dark_channel(image, size=15):
    min_channel = np.min(image, axis=2)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (size, size))
    return cv2.erode(min_channel, kernel)

def apply_dehaze(frame):
    img_f = frame.astype(np.float32) / 255.0
    dark = get_dark_channel(img_f, size=15)
    A = np.percentile(dark, 99)
    t = 1.0 - 0.95 * dark
    t = np.clip(t, 0.1, 1.0)
    J = (img_f - A) / cv2.merge([t, t, t]) + A
    J = np.clip(J, 0, 1)
    return (J * 255).astype(np.uint8)

def apply_clahe(frame):
    img_yuv = cv2.cvtColor(frame, cv2.COLOR_BGR2YUV)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    img_yuv[:,:,0] = clahe.apply(img_yuv[:,:,0])
    return cv2.cvtColor(img_yuv, cv2.COLOR_YUV2BGR)

def apply_adaptive_gamma(frame):
    img_yuv = cv2.cvtColor(frame, cv2.COLOR_BGR2YUV)
    y = img_yuv[:, :, 0]
    mean_bright = np.mean(y) + 1e-5
    gamma = np.log(128/255) / np.log(mean_bright/255)
    gamma = np.clip(gamma, 0.5, 2.5)
    invGamma = 1.0 / gamma
    table = np.array([((i / 255.0) ** invGamma) * 255 for i in np.arange(0, 256)]).astype("uint8")
    img_yuv[:, :, 0] = cv2.LUT(y, table)
    return cv2.cvtColor(img_yuv, cv2.COLOR_YUV2BGR)

def apply_bilateral_denoise(frame):
    return cv2.bilateralFilter(frame, 9, 75, 75)

# Enhancement Pipelines
ENHANCEMENT_PIPELINES = {
    'none': lambda x: x,
    'dehaze_only': lambda x: apply_dehaze(x),
    'clahe_only': lambda x: apply_clahe(x),
    'gamma_clahe': lambda x: apply_clahe(apply_adaptive_gamma(x)),
    'dehaze_clahe': lambda x: apply_clahe(apply_dehaze(x)),
    'denoise_clahe': lambda x: apply_clahe(apply_bilateral_denoise(x)),
    'full_recovery': lambda x: apply_clahe(apply_adaptive_gamma(apply_bilateral_denoise(apply_dehaze(x)))),
}

# Stress Scenarios
STRESS_SCENARIOS = {
    'pristine': {'darkness': 0.0, 'fog': 0.0, 'noise': 0.0, 'blur': 0.0},
    'mild_darkness': {'darkness': 0.3, 'fog': 0.0, 'noise': 0.0, 'blur': 0.0},
    'severe_darkness': {'darkness': 0.6, 'fog': 0.0, 'noise': 0.0, 'blur': 0.0},
    'extreme_darkness': {'darkness': 0.8, 'fog': 0.0, 'noise': 0.0, 'blur': 0.0},
    'mild_fog': {'darkness': 0.0, 'fog': 0.3, 'noise': 0.0, 'blur': 0.0},
    'dense_fog': {'darkness': 0.0, 'fog': 0.6, 'noise': 0.0, 'blur': 0.0},
    'extreme_fog': {'darkness': 0.0, 'fog': 0.8, 'noise': 0.0, 'blur': 0.0},
    'mild_noise': {'darkness': 0.0, 'fog': 0.0, 'noise': 0.3, 'blur': 0.0},
    'heavy_noise': {'darkness': 0.0, 'fog': 0.0, 'noise': 0.6, 'blur': 0.0},
    'mild_blur': {'darkness': 0.0, 'fog': 0.0, 'noise': 0.0, 'blur': 0.3},
    'severe_blur': {'darkness': 0.0, 'fog': 0.0, 'noise': 0.0, 'blur': 0.6},
    'night_fog': {'darkness': 0.5, 'fog': 0.4, 'noise': 0.2, 'blur': 0.0},
    'rainy_night': {'darkness': 0.6, 'fog': 0.3, 'noise': 0.3, 'blur': 0.2},
    'extreme_conditions': {'darkness': 0.7, 'fog': 0.5, 'noise': 0.4, 'blur': 0.3},
}

def analyze_stressed_video(video_path, degradation_params, enhancement_method='none', frame_limit=FRAME_LIMIT):
    cap = cv2.VideoCapture(video_path)
    tracker = DeepSort(max_age=50)
    model = YOLO("yolov8n.pt")
    
    track_ids = set()
    total_frames_tracked = 0
    confidences = []
    detections_per_frame = []
    brightness_values = []
    enhancement_func = ENHANCEMENT_PIPELINES[enhancement_method]
    
    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret or frame_idx > frame_limit:
            break
        
        degraded = apply_combined_degradation(frame, **degradation_params)
        processed = enhancement_func(degraded)
        
        gray = cv2.cvtColor(processed, cv2.COLOR_BGR2GRAY)
        brightness_values.append(np.mean(gray))
        
        results = model(processed, verbose=False)
        dets = []
        for r in results:
            for box in r.boxes:
                if float(box.conf[0]) > CONF_THRESHOLD:
                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    conf = float(box.conf[0])
                    dets.append([[x1, y1, x2-x1, y2-y1], conf, int(box.cls[0])])
                    confidences.append(conf)
        
        detections_per_frame.append(len(dets))
        tracks = tracker.update_tracks(dets, frame=processed)
        for t in tracks:
            if t.is_confirmed():
                track_ids.add(t.track_id)
                total_frames_tracked += 1
        frame_idx += 1
    
    cap.release()
    unique_ids = len(track_ids)
    
    return {
        'Unique IDs': unique_ids,
        'Avg Duration': total_frames_tracked / unique_ids if unique_ids > 0 else 0,
        'Avg Confidence': np.mean(confidences) if confidences else 0,
        'Detections/Frame': np.mean(detections_per_frame),
        'Total Detections': sum(detections_per_frame),
        'Avg Brightness': np.mean(brightness_values)
    }

# Main Execution Loop
video_files = [f for f in os.listdir(INPUT_DIR) if f.endswith(('.avi', '.mp4'))]
video_files = [f for f in video_files if "result" not in f and "stress" not in f][:2]

all_results = []
for vid_idx, vid in enumerate(video_files):
    video_path = os.path.join(INPUT_DIR, vid)
    for scenario_name, deg_params in STRESS_SCENARIOS.items():
        for enh_method in ENHANCEMENT_PIPELINES.keys():
            result = analyze_stressed_video(video_path, deg_params, enh_method, frame_limit=FRAME_LIMIT)
            result.update({
                'Video': vid,
                'Scenario': scenario_name,
                'Enhancement': enh_method,
                'Degradation_Darkness': deg_params['darkness'],
                'Degradation_Fog': deg_params['fog'],
                'Degradation_Noise': deg_params['noise'],
                'Degradation_Blur': deg_params['blur']
            })
            all_results.append(result)

df_stress = pd.DataFrame(all_results)
df_stress.to_csv(os.path.join(OUTPUT_DIR, "stress_test_detailed.csv"), index=False)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

INPUT_CSV = "stress_test_detailed.csv"
OUTPUT_DIR = "content/final_analysis"
os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv('content/processed/stress_tests/stress_test_detailed.csv')

# 1. Crossover Plot
plt.figure(figsize=(12, 6))
sns.barplot(data=df, x='Scenario', y='Detections/Frame', hue='Enhancement', palette='viridis')
plt.title("The 'Crossover Effect': Where Enhancements Become Essential", fontsize=14, fontweight='bold')
plt.ylabel("Vehicle Detections Per Frame")
plt.xlabel("Environmental Condition")
plt.legend(title="Method", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "final_crossover_effect.png"))

# 2. Efficiency Analysis
pristine_df = df[df['Scenario'] == 'pristine']
plt.figure(figsize=(10, 5))
sns.scatterplot(data=pristine_df, x='Total Detections', y='Avg Duration', hue='Enhancement', s=200, style='Enhancement')
plt.title("Pristine Conditions: Cost vs Benefit", fontsize=14, fontweight='bold')
plt.xlabel("Total Detections (Recall)")
plt.ylabel("Avg Track Duration (Stability)")
plt.grid(True, linestyle='--')
plt.savefig(os.path.join(OUTPUT_DIR, "pristine_tradeoff.png"))

# 3. Summary Table
summary = df.groupby(['Scenario', 'Enhancement'])[['Detections/Frame', 'Avg Confidence', 'Avg Duration']].mean()
summary.to_csv(os.path.join(OUTPUT_DIR, "final_conclusive_table.csv"))

In [ ]:
import cv2
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
from deep_sort_realtime.deepsort_tracker import DeepSort
from ultralytics import YOLO
from itertools import product

# Scene Analysis Functions
def estimate_brightness(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    return np.mean(gray)

def estimate_fog_level(frame):
    dark_channel = np.min(frame, axis=2)
    fog_metric = 1.0 - (np.mean(dark_channel) / 255.0)
    return fog_metric

def analyze_scene(frame):
    return {
        'brightness': estimate_brightness(frame),
        'fog_level': estimate_fog_level(frame)
    }

# Adaptive Pipeline Strategies
class AdaptivePipeline:
    def __init__(self, name, description):
        self.name = name
        self.description = description
    
    def process(self, frame, scene_params):
        raise NotImplementedError

class Strategy_Fog_Aware(AdaptivePipeline):
    def __init__(self, fog_threshold=0.5, omega=0.95):
        super().__init__("fog_aware", f"Dehaze if fog>{fog_threshold}")
        self.fog_threshold = fog_threshold
        self.omega = omega
    
    def process(self, frame, scene_params):
        if scene_params['fog_level'] > self.fog_threshold:
            return apply_dehaze(frame, omega=self.omega) # Uses function from Block 1
        return frame

class Strategy_Darkness_Aware(AdaptivePipeline):
    def __init__(self, brightness_threshold=100, clip_limit=2.5):
        super().__init__("darkness_aware", f"CLAHE if bright<{brightness_threshold}")
        self.brightness_threshold = brightness_threshold
        self.clip_limit = clip_limit
    
    def process(self, frame, scene_params):
        if scene_params['brightness'] < self.brightness_threshold:
            return apply_clahe(frame) # Uses function from Block 1
        return frame

class Strategy_Hybrid_Adaptive(AdaptivePipeline):
    def __init__(self, fog_thresh=0.5, dark_thresh=100, fog_omega=0.95, clahe_clip=2.0):
        super().__init__("hybrid_adaptive", "Fog>Thresh:Dehaze, Dark<Thresh:CLAHE")
        self.fog_thresh = fog_thresh
        self.dark_thresh = dark_thresh
        self.fog_omega = fog_omega
        self.clahe_clip = clahe_clip
    
    def process(self, frame, scene_params):
        if scene_params['fog_level'] > self.fog_thresh:
            dehazed = apply_dehaze(frame)
            if scene_params['brightness'] < self.dark_thresh:
                return apply_clahe(dehazed)
            return dehazed
        elif scene_params['brightness'] < self.dark_thresh:
            return apply_clahe(frame)
        return frame

# Video Analysis Engine for Adaptive Strategies
def analyze_video_adaptive(video_path, strategy, degradation_params=None, frame_limit=150):
    cap = cv2.VideoCapture(video_path)
    tracker = DeepSort(max_age=50)
    model = YOLO("yolov8n.pt")
    
    confidences = []
    detections = []
    
    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret or frame_idx > frame_limit: break
        
        if degradation_params:
            frame = apply_combined_degradation(frame, **degradation_params)
        
        scene_params = analyze_scene(frame)
        processed = strategy.process(frame, scene_params)
        
        results = model(processed, verbose=False)
        for r in results:
            for box in r.boxes:
                if float(box.conf[0]) > 0.50:
                    confidences.append(float(box.conf[0]))
                    detections.append(1)
        frame_idx += 1
        
    cap.release()
    return {
        'Avg Confidence': np.mean(confidences) if confidences else 0,
        'Total Detections': sum(detections)
    }

# Threshold Optimization
def optimize_thresholds(video_path, strategy_class, param_ranges, degradation_params=None):
    param_names = list(param_ranges.keys())
    combinations = list(product(*param_ranges.values()))
    results = []
    
    for combo in combinations:
        params = dict(zip(param_names, combo))
        strategy = strategy_class(**params)
        result = analyze_video_adaptive(video_path, strategy, degradation_params)
        result['params'] = str(params)
        results.append(result)
    
    df = pd.DataFrame(results)
    best_result = df.loc[df['Avg Confidence'].idxmax()]
    return df, best_result

# --- FINAL PRODUCTION CODE ---
def production_adaptive_enhancement(frame):
    # Optimized adaptive pipeline with data-driven thresholds
    brightness = np.mean(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY))
    dark_channel = np.min(frame, axis=2)
    fog_level = 1.0 - (np.mean(dark_channel) / 255.0)
    
    # Thresholds determined by optimization (Block 3 results)
    FOG_THRESHOLD = 0.50
    BRIGHTNESS_THRESHOLD = 100
    
    if fog_level > FOG_THRESHOLD:
        # Fog detected: apply dehaze
        img_f = frame.astype(np.float32) / 255.0
        dark = np.min(img_f, axis=2)
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
        dark = cv2.erode(dark, kernel)
        A = np.percentile(dark, 99)
        t = 1.0 - 0.95 * dark
        t = np.clip(t, 0.1, 1.0)
        J = (img_f - A) / np.expand_dims(t, axis=2) + A
        J = np.clip(J, 0, 1)
        enhanced = (J * 255).astype(np.uint8)
        
        # If also dark, add CLAHE
        if brightness < BRIGHTNESS_THRESHOLD:
            img_yuv = cv2.cvtColor(enhanced, cv2.COLOR_BGR2YUV)
            clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8,8))
            img_yuv[:,:,0] = clahe.apply(img_yuv[:,:,0])
            return cv2.cvtColor(img_yuv, cv2.COLOR_YUV2BGR)
        return enhanced
    
    elif brightness < BRIGHTNESS_THRESHOLD:
        # Dark conditions: apply CLAHE
        img_yuv = cv2.cvtColor(frame, cv2.COLOR_BGR2YUV)
        clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8,8))
        img_yuv[:,:,0] = clahe.apply(img_yuv[:,:,0])
        return cv2.cvtColor(img_yuv, cv2.COLOR_YUV2BGR)
    
    return frame

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

data = {
    'Scenario': ['Pristine', 'Pristine', 'Extreme Fog', 'Extreme Fog', 'Severe Darkness', 'Severe Darkness'],
    'Pipeline': ['Baseline', 'Adaptive (Ours)', 'Baseline', 'Adaptive (Ours)', 'Baseline', 'Adaptive (Ours)'],
    'Confidence Score': [0.615, 0.615, 0.311, 0.629, 0.590, 0.631]
}

df = pd.DataFrame(data)

# Calculate improvements
improvement = {}
scenarios = ["Pristine", "Extreme Fog", "Severe Darkness"]
for s in scenarios:
    base = df[(df["Scenario"] == s) & (df["Pipeline"] == "Baseline")]["Confidence Score"].values[0]
    adapt = df[(df["Scenario"] == s) & (df["Pipeline"] == "Adaptive (Ours)")]["Confidence Score"].values[0]
    improvement[s] = adapt - base

# Plot
plt.figure(figsize=(10, 6))
sns.set_theme(style="whitegrid")
palette = {"Baseline": "#7f8c8d", "Adaptive (Ours)": "#27ae60"}

ax = sns.barplot(
    data=df, x='Scenario', y='Confidence Score', hue='Pipeline',
    palette=palette, edgecolor="black", linewidth=1.5
)

# Annotations
annotation_positions = {"Pristine": (-0.05, 0.63), "Extreme Fog": (1, 0.66), "Severe Darkness": (2.05, 0.66)}

for s, (x, y) in annotation_positions.items():
    imp = improvement[s]
    label = f"+{imp:.3f}" if abs(imp) > 0.001 else "No Change"
    plt.text(x, y, label, ha='center', va='bottom',
             color='green' if imp > 0 else 'red', fontweight='bold', fontsize=10)

plt.title("Impact of Adaptive Processing vs. Baseline", fontsize=16, fontweight='bold')
plt.ylabel("YOLO Detection Confidence", fontsize=12, fontweight='bold')
plt.xlabel("Environmental Condition", fontsize=12, fontweight='bold')
plt.ylim(0, 0.8)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()